In [1]:
import pandas as pd

df = pd.read_csv("data/dublin_clean.csv")
df.shape

(2499, 18)

In [2]:
# splitting the comma-separated strings into clean lists of individual items
df["cuisines_list"] = df["cuisines"].fillna("").str.split(", ")
df["tags_list"] = df["top_tags"].fillna("").str.split(", ")

# checking the result on the first few rows
df[["cuisines", "cuisines_list", "top_tags", "tags_list"]].head()

,cuisines,cuisines_list,top_tags,tags_list
0,Cafe,[Cafe],"Cheap Eats, Cafe","[Cheap Eats, Cafe]"
1,"Indian, Asian","[Indian, Asian]","Mid-range, Indian, Asian, Vegetarian Friendly","[Mid-range, Indian, Asian, Vegetarian Friendly]"
2,American,[American],"Cheap Eats, American, Gluten Free Options","[Cheap Eats, American, Gluten Free Options]"
3,"Irish, European, Contemporary","[Irish, European, Contemporary]","Mid-range, Irish, European, Contemporary","[Mid-range, Irish, European, Contemporary]"
4,Cafe,[Cafe],"Mid-range, Cafe, Vegetarian Friendly, Vegan Op...","[Mid-range, Cafe, Vegetarian Friendly, Vegan O..."


In [3]:
# combine cuisines & tags into one text "profile" per restaurant
df["combined_features"] = (df["cuisines"].fillna("") + ", " + df["top_tags"].fillna(""))

# checking a few rows
df[["restaurant_name", "combined_features"]].head()

,restaurant_name,combined_features
0,Vape Caffè,"Cafe, Cheap Eats, Cafe"
1,Pickle,"Indian, Asian, Mid-range, Indian, Asian, Veget..."
2,Bunsen South Anne Street,"American, Cheap Eats, American, Gluten Free Op..."
3,Craft,"Irish, European, Contemporary, Mid-range, Iris..."
4,U Cafe,"Cafe, Mid-range, Cafe, Vegetarian Friendly, Ve..."


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create the vectoriser & turn the text profiles into a numeric matrix
tfidf = TfidfVectorizer(token_pattern=r"[^,]+")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

# checking how big the result is
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of features (unique terms):", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (2499, 178)
Number of features (unique terms): 178


In [5]:
# check the terms TF-IDF learned (confirm that the multi-word tags stayed intact)
feature_names = tfidf.get_feature_names_out()
print(feature_names[:40])

[' afghani' ' african' ' albanian' ' algerian' ' american' ' apulian'
 ' arabic' ' argentinian' ' armenian' ' asian' ' assyrian' ' australian'
 ' azerbaijani' ' bakeries' ' balti' ' bangladeshi' ' bar' ' barbecue'
 ' beer restaurants' ' belgian' ' brazilian' ' brew pub' ' british'
 ' cafe' ' cajun & creole' ' calabrian' ' campania' ' canadian'
 ' cantonese' ' caribbean' ' catalan' ' central american' ' central asian'
 ' central european' ' central-italian' ' cheap eats' ' chinese'
 ' colombian' ' contemporary' ' croatian']


In [6]:
# rebuild combined_features with clean spacing (strip stray spaces around each term)

def clean_join(row):    
    items = (str(row["cuisines"]) + ", " + str(row["top_tags"])).split(",")
    items = [item.strip().lower() for item in items if item.strip() not in ["", "nan"]]
    return ",".join(items)

df["combined_features"] = df.apply(clean_join, axis=1)

# rebuild the TF-IDF matrix on the cleaned text
tfidf = TfidfVectorizer(token_pattern=r"[^,]+")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print(tfidf.get_feature_names_out()[:20])

TF-IDF matrix shape: (2499, 124)
['afghani' 'african' 'albanian' 'algerian' 'american' 'apulian' 'arabic'
 'argentinian' 'armenian' 'asian' 'assyrian' 'australian' 'azerbaijani'
 'bakeries' 'balti' 'bangladeshi' 'bar' 'barbecue' 'beer restaurants'
 'belgian']


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# compute similarity between every restaurant & every other restaurant
cosine_sim = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", cosine_sim.shape)

Similarity matrix shape: (2499, 2499)


In [8]:
def recommend_similar(restaurant_name, top_n=5):
    # find the row index of the given restaurant
    idx = df[df["restaurant_name"] == restaurant_name].index[0]

    # get its similarity scores against all restaurants
    scores = list(enumerate(cosine_sim[idx]))

    # sort by similarity, highest first
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # skip the first one (it's the restaurant itself, score 1.0) and take the next top_n
    top_matches = scores[1:top_n+1]

    # build a readable result
    results = []
    for i, score in top_matches:
        results.append({
            "name": df.iloc[i]["restaurant_name"],
            "cuisines": df.iloc[i]["cuisines"],
            "similarity": round(score, 3)
        })
    return pd.DataFrame(results)

# testing with "Pickle"
recommend_similar("Pickle")

,name,cuisines,similarity
0,Sitar Indian Cuisine,"Indian, Asian",1.0
1,Sitar Restaurant Temple Bar,"Indian, Asian",1.0
2,Chutni Indian,"Indian, Asian",1.0
3,Chaska,"Indian, Asian",1.0
4,INDIA TODAY Bar & Grill,"Indian, Asian",1.0


In [9]:
# testing with "Craft"
recommend_similar("Craft")

,name,cuisines,similarity
0,Restaurant SIX,"Irish, European, Contemporary",1.0
1,Layla's,"European, Irish, Contemporary",1.0
2,The Grandstand Bar & Restaurant,"Irish, European, Contemporary",1.0
3,Camden Kitchen,"Irish, European, Contemporary",1.0
4,The Cookbook Cafe,"Irish, Contemporary, European",1.0


In [10]:
# preference-based recommender

def recommend_for_user(preferences, top_n=5):
    # preferences: a list of things the user likes, --> ["italian", "mid-range"]
    # then turn them into the same comma-joined format our TF-IDF was trained on
    user_text = ",".join([p.strip().lower() for p in preferences])

    # transform the user's text into a vector using the *same* fitted tfidf
    user_vector = tfidf.transform([user_text])

    # compare the user's vector to every restaurant
    scores = cosine_similarity(user_vector, tfidf_matrix)[0]

    # get the indices of the top matches, highest first
    top_idx = scores.argsort()[::-1][:top_n]

    # build a readable result table (FR-004: name, cuisine, price, rating)
    results = []
    for i in top_idx:
        results.append({
            "name": df.iloc[i]["restaurant_name"],
            "cuisines": df.iloc[i]["cuisines"],
            "price_level": df.iloc[i]["price_level"],
            "avg_rating": df.iloc[i]["avg_rating"],
            "similarity": round(scores[i], 3)
        })
    return pd.DataFrame(results)

# TEST
# a user who wants Italian & mid-range
recommend_for_user(["italian", "mid-range"])

,name,cuisines,price_level,avg_rating,similarity
0,Nero XVII,Italian,€€-€€€,5.0,0.98
1,Bell Pesto Cafè,Italian,€€-€€€,5.0,0.98
2,Belle Cafe,Italian,€€-€€€,NaN,0.98
3,Romanza,Italian,€€-€€€,NaN,0.98
4,Glad Café,Italian,€€-€€€,3.0,0.98


In [11]:
# verify NFR-001: the recommendation returns in under 2 seconds

import time

start = time.time()
recommend_for_user(["italian", "mid-range"])
print(f"Recommendation time: {time.time() - start:.4f} seconds")

Recommendation time: 0.0021 seconds
